In [25]:
import os
os.environ["GOOGLE_API_KEY"] = "AQ.Ab8RN6LOpTgG3l-rG8V3eREBYlL2boQ7lfeZ2QO72A4ApGvV0g"

In [26]:
#!pip install langchain-text-splitters

In [67]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

## Document Ingestion

In [48]:
video_id = "m7v7KLttM3k" # only the ID, not full URL
try:
    # If you don’t care which language, this returns the “best” one
    api = YouTubeTranscriptApi()
    transcript_list = api.fetch(video_id, languages=["en"])

    # Flatten it to plain text
    transcript = " ".join(chunk.text for chunk in transcript_list)
    print(transcript)

except TranscriptsDisabled:
    print("No captions available for this video.")

At one point, al-Qaeda came close to killing you. Like you were shot eight times. Yeah. Have your skull. Walk me through what happened. I was fascinated by the west that you got. I want to talk about the details of the west. How much would it weigh? >> I mean, fully loaded anywhere from 25 to 30 lb. >> If you get a bullet from right where you sit, will this protect me? >> I mean, if it was a handgun, 100%. And sniper can get through this, you'd probably aim for your head. Jason Redmond, retired US Navy Sea Lieutenant. He served across combat deployments in Iraq and Afghanistan, leading elite special operations teams in some of the world's most dangerous environments. >> Tell me about your time in Afghanistan. >> We were going after a senior al-Qaeda leader, the first target we thought he was going to be in, and no one was there. And then we saw some activity on another building about 150 yards away. I took my team and we moved on that building and we were moving through some really thi

## Text Splitting

In [49]:
splitter= RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript])

In [50]:
len(chunks)

104

In [51]:
chunks[100]

Document(metadata={}, page_content='of Defense Robert Gates wrote about it in his book. First Lady Michelle Obama wrote about it twice in her book and sent me a handwritten note on how much it moved her. I tell people, "What do you want your sign on the door to say?" Everyone has their own sign on the door inside them. and and maybe it\'s not as elaborate as this, but it\'s about who you are and and a reminder so that in the hard times, it keeps you grounded and it keeps you moving forward towards your hopes, goals, and dreams. So that this is my new mission. And and most people have never written it down. I\'ll ask them, "Have you ever written down a mission statement?" And they\'re like, "No, but I know what mine is. I don\'t need to write it down." And I\'m like, I disagree. Cuz humans, we have short-term memory. And when we\'re in pain or uncomfortable, we forget. That\'s why so many people quit SEAL training. Their goal, they came there to be a Navy Seal. But when they\'re in pain

In [59]:
small_chunks = chunks[:100]

## Embedding Generation and Storing in Vector Store

In [60]:
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
vector_store = FAISS.from_documents(small_chunks,embeddings)

In [61]:
vector_store.index_to_docstore_id

{0: 'c14e8dda-742f-4f44-9a8b-9d9e1bae90fb',
 1: '9a4df674-cd7a-4c5a-81f2-9c1ac12f984c',
 2: '54a25e0c-272b-4e31-abc7-e8b2a5b85735',
 3: '88b919e1-d45e-455b-979e-8ff78685a70c',
 4: '0dc3a132-f2a6-4997-86f5-17b2ccf3b636',
 5: '45efadc3-8f79-47b4-b874-8d71678227af',
 6: '75ff961d-de7c-4f32-910e-e70e9e45cccb',
 7: 'dd826773-fe8f-4406-a0f4-78ea59013500',
 8: '5ccafe03-6762-4b72-b72b-f4b26ebdd8ab',
 9: '0194454e-3309-49bb-96d7-e274bccfa0c9',
 10: '7e6571b4-1ab1-49a1-8a86-69587c2a6ebc',
 11: '446836e6-12cd-4b28-83a4-41be801893b8',
 12: 'd9ff975c-b473-425d-905e-113715c770c6',
 13: '6eaf77ee-ef69-4e5b-a346-4076f8e3efd0',
 14: '0e9acecb-0e73-4020-aa92-e9b5d03e261a',
 15: '724e913c-8cd1-40b9-9ef6-913a733e5685',
 16: '1fcf4233-c53a-4488-990d-de3af7ed65c7',
 17: 'e5e4aefc-5a98-4b26-be9f-40826cadbb32',
 18: '51ede580-a4e1-41ec-aa01-915fa4af7e6f',
 19: '0e361b46-9dd3-412b-a932-9b719bf338c1',
 20: 'e54f3b75-eb47-48b6-9903-483349062307',
 21: 'd549f7c5-617c-4b97-838f-5670381844ab',
 22: '60999059-f498-

In [62]:
vector_store.get_by_ids(['f89152a1-8d0f-46d4-92a0-13f5ffefd378'])

[Document(id='f89152a1-8d0f-46d4-92a0-13f5ffefd378', metadata={}, page_content='I didn\'t think about like, hey, it\'s our job to go get bad guys. That sure didn\'t keep me alive. I thought about my wife and kids. It\'s your own personal mission that will make the difference in this life. That\'ll help you navigate. I think that\'s what will enable this balance of personal success and and and professional or financial success. And I and I and that sign um it has guided me now. And even today I look at it and I\'m like, "Hey man, that\'s who you are." You know, >> can you read that for me? >> Yeah. It says, "Attention to all who enter here. If you\'re coming in this room with sadness or sorrow, don\'t bother. The wounds I received, I got a job that I love, doing it for people that I love, defending the freedom of a country that I deeply love. I will make a full recovery. What is full? That is the absolute utmost physically. I have the ability to recover. And then I\'m going to push that

## Retrieval

In [63]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [64]:
retriever

VectorStoreRetriever(tags=['FAISS', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000019950A3CF50>, search_kwargs={'k': 4})

In [65]:
retriever.invoke('What is hell week?')

[Document(id='724e913c-8cd1-40b9-9ef6-913a733e5685', metadata={}, page_content="like 60% of the time. In hell week, you're wet and sandy 95% of the time. You're almost never dry. >> What happens to the entire week? Tell me like step by step. The hell week, >> uh it starts with something called breakout where they uh they it simulates combat. It just simulates there's uh there's gunfire and explosions and and just chaos. Uh and then the week goes from there and there's just you're you're always um we have these rubber boats. You're in a boat crew and those rubber boats everywhere you go you're carrying them on your head. Uh those boats weigh probably 250 300 lb dry but because you're constantly in the water and on the beach they get filled with sand. And so they probably end up weighing 400 lb or so. Um, so uh and and the sand gets in between your head and the bottom of the boat. So you're you have a sore on top of your head usually within a couple of days. Your neck is aching because y

## Augmentation

In [68]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash",temperature=0.2)

In [69]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [76]:
question          = "is the topic of navy seals training discussed in this video? if yes then what was discussed"
retrieved_docs    = retriever.invoke(question)

In [77]:
retrieved_docs

[Document(id='7e6571b4-1ab1-49a1-8a86-69587c2a6ebc', metadata={}, page_content="freedom is a very powerful thing. and this idea of freedom and and it and it is it is born out of a willingness to fight. So if you were raised that way, which I was, um I was born in 1975, so that was only um 30 years after World War II. That was very fresh in the minds of my parents and my grandparents. So my my both grandfathers fought in World War II. So as a young man I was raised on these stories. >> But then when you went to the army you opted to become Navy Seal. >> I did. >> Right. That training is hard. It's it just can't be done because you aspired to be into the army and you watch GI Joe and you wanted to actually be in there, right? That that must be brutal. >> It is. Uh yeah, I mean it is considered the hardest training in the entire United States military. We have a uh 80% attrition rate. So uh only two out of 10 people that start training graduate. >> So tell me about the training routine en

In [78]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

"freedom is a very powerful thing. and this idea of freedom and and it and it is it is born out of a willingness to fight. So if you were raised that way, which I was, um I was born in 1975, so that was only um 30 years after World War II. That was very fresh in the minds of my parents and my grandparents. So my my both grandfathers fought in World War II. So as a young man I was raised on these stories. >> But then when you went to the army you opted to become Navy Seal. >> I did. >> Right. That training is hard. It's it just can't be done because you aspired to be into the army and you watch GI Joe and you wanted to actually be in there, right? That that must be brutal. >> It is. Uh yeah, I mean it is considered the hardest training in the entire United States military. We have a uh 80% attrition rate. So uh only two out of 10 people that start training graduate. >> So tell me about the training routine entire routine like from day one to how many phases is the training? What goes\n\

In [79]:
final_prompt = prompt.invoke({"context": context_text, "question": question})
final_prompt

StringPromptValue(text="\n      You are a helpful assistant.\n      Answer ONLY from the provided transcript context.\n      If the context is insufficient, just say you don't know.\n\n      freedom is a very powerful thing. and this idea of freedom and and it and it is it is born out of a willingness to fight. So if you were raised that way, which I was, um I was born in 1975, so that was only um 30 years after World War II. That was very fresh in the minds of my parents and my grandparents. So my my both grandfathers fought in World War II. So as a young man I was raised on these stories. >> But then when you went to the army you opted to become Navy Seal. >> I did. >> Right. That training is hard. It's it just can't be done because you aspired to be into the army and you watch GI Joe and you wanted to actually be in there, right? That that must be brutal. >> It is. Uh yeah, I mean it is considered the hardest training in the entire United States military. We have a uh 80% attrition 

## Generation

In [80]:
answer = llm.invoke(final_prompt)
print(answer.content)

Yes, the topic of Navy SEALs training is discussed in the provided transcript.

Here's what was discussed about the training:
*   It is considered the hardest training in the entire United States military.
*   It has an 80% attrition rate, meaning only two out of ten people who start graduate.
*   The initial qualification training is 7 months long and broken into three phases.
*   The first phase is designed to break participants mentally.
*   During the first phase, trainees are often wet and coated in sand, leading to constant chafing and being freezing cold from the water in California.
*   This phase culminates in "hell week," which is notorious and considered the hardest week of training in the US military.
*   During hell week, trainees are not allowed to sleep for a week, are constantly moving, and are wet and sandy 95% of the time.
*   The training teaches participants to focus and accomplish tasks even when uncomfortable and in pain.
*   Some individuals, despite being physic

## Building a chain

In [81]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [82]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [83]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [84]:
parallel_chain.invoke('who is Jason Redman')

{'context': 'At one point, al-Qaeda came close to killing you. Like you were shot eight times. Yeah. Have your skull. Walk me through what happened. I was fascinated by the west that you got. I want to talk about the details of the west. How much would it weigh? >> I mean, fully loaded anywhere from 25 to 30 lb. >> If you get a bullet from right where you sit, will this protect me? >> I mean, if it was a handgun, 100%. And sniper can get through this, you\'d probably aim for your head. Jason Redmond, retired US Navy Sea Lieutenant. He served across combat deployments in Iraq and Afghanistan, leading elite special operations teams in some of the world\'s most dangerous environments. >> Tell me about your time in Afghanistan. >> We were going after a senior al-Qaeda leader, the first target we thought he was going to be in, and no one was there. And then we saw some activity on another building about 150 yards away. I took my team and we moved on that building and we were moving through 

In [85]:
parser = StrOutputParser()

In [86]:
main_chain = parallel_chain | prompt | llm | parser

In [87]:
main_chain.invoke('Can you summarize the video')

'The transcript discusses several topics, including:\n\n*   The nature of war, the threat posed by Iran\'s commitment to developing nuclear or chemical/biological weapons, and the belief that America can win but will require ground forces.\n*   The speaker\'s gratitude for being on the channel and his self-introduction as an "average everyday human" from a family that believed in their country.\n*   The concept of hitting a "breaking point" and the choice not to quit, contrasting it with the majority who do quit.\n*   An explanation of "violence of action" in combat as a flurry of violent activity designed to overwhelm an opponent.\n*   The idea that even powerful nations struggle against the "human spirit" in smaller countries, and the speaker\'s strong belief in the Second Amendment as a check against tyrannical regimes, noting the lack of weapons among the Iranian people to overthrow their government.'